# Phase 5 — Fairness Audit

A model that performs well on average can still harm specific patient groups.
This audit examines whether our model treats patients equitably across demographic
groups. We focus on three dimensions: **ethnicity**, **gender**, and **age group**.

Clinical relevance is high: Obermeyer et al. (Science, 2019) showed that
cost-based risk algorithms systematically under-flagged Black patients; Sjoding et al.
(NEJM, 2020) demonstrated that pulse oximeters over-read SpO₂ in dark-skinned
patients, biasing any model trained on that signal. High overall AUC does **not**
guarantee equitable performance across subgroups.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('../'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.models import load_model
from src.preprocessing import scale_numeric
from src.fairness import (
    run_metric_frame, summarise_disparities, plot_metric_by_group,
    compute_subgroup_metrics, plot_grouped_bar_charts,
    bootstrap_group_diff, plot_metricframe_heatmap, run_threshold_optimizer,
)

os.makedirs('../reports/figures', exist_ok=True)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

# ── Load model ──────────────────────────────────────────────────────────────
model = load_model('lgbm_best')
print(f'Model loaded: {type(model).__name__}')

# ── Reconstruct Phase 3 test split (same indices guarantee comparability) ──
df = pd.read_csv('../data/processed/features_engineered.csv')
X_all = df.drop(columns=['hospital_death'])
y_all = df['hospital_death']

split_path = Path('../data/processed/split_indices.npz')
if split_path.exists():
    indices    = np.load(split_path)
    train_idx  = indices['train_idx']
    val_idx    = indices['val_idx']
    test_idx   = indices['test_idx']
    print(f'Split indices loaded. Test n={len(test_idx):,}')
else:
    from sklearn.model_selection import train_test_split
    train_val_idx, test_idx = train_test_split(
        np.arange(len(df)), test_size=0.15, random_state=42, stratify=y_all)
    train_idx, val_idx = train_test_split(
        train_val_idx, test_size=0.15/0.85, random_state=42,
        stratify=y_all.iloc[train_val_idx])
    print('WARNING: split_indices.npz not found — re-splitting. '
          'Results may differ slightly from Phase 3.')

X_train_raw = X_all.iloc[train_idx]
X_val_raw   = X_all.iloc[val_idx]
X_test_raw  = X_all.iloc[test_idx].reset_index(drop=True)
y_train = y_all.iloc[train_idx].reset_index(drop=True)
y_val   = y_all.iloc[val_idx].reset_index(drop=True)
y_test  = y_all.iloc[test_idx].reset_index(drop=True)

X_train_s, X_val_s, X_test_s, _ = scale_numeric(X_train_raw, X_val_raw, X_test_raw)

# ── Predictions ─────────────────────────────────────────────────────────────
OPTIMAL_THRESHOLD = 0.35   # conservative threshold from Phase 3 threshold optimisation
X_test_s = X_test_s.select_dtypes(include=['number'])
y_prob = model.predict_proba(X_test_s)[:, 1]
y_pred = (y_prob >= OPTIMAL_THRESHOLD).astype(int)
print(f'Threshold: {OPTIMAL_THRESHOLD}')
print(f'Predicted positive rate: {y_pred.mean():.4f}')
print(f'Actual positive rate:    {y_test.mean():.4f}')

# ── Build test_df with sensitive attributes ──────────────────────────────────
test_df = X_test_raw.copy()
test_df['hospital_death'] = y_test.values
test_df['y_pred']         = y_pred
test_df['y_prob']         = y_prob

# age_group: prefer the engineered column, fall back to re-binning
if 'age_group' not in test_df.columns and 'age' in test_df.columns:
    AGE_BINS   = [-np.inf, 45, 65, 80, np.inf]
    AGE_LABELS = ['young_adult', 'middle_aged', 'older_adult', 'elderly']
    test_df['age_group'] = pd.cut(
        test_df['age'], bins=AGE_BINS, labels=AGE_LABELS, right=False
    ).astype(str)

# collapse rare ethnicity categories (< 50 patients) into 'Other/Unknown'
if 'ethnicity' in test_df.columns:
    eth_counts = test_df['ethnicity'].value_counts()
    rare = eth_counts[eth_counts < 50].index
    test_df['ethnicity_grp'] = test_df['ethnicity'].apply(
        lambda x: 'Other/Unknown' if x in rare else x
    )
else:
    test_df['ethnicity_grp'] = 'Unknown'

# normalise gender to string
if 'gender' in test_df.columns:
    test_df['gender'] = test_df['gender'].astype(str)

SENSITIVE_ATTRS = [a for a in ['ethnicity_grp', 'gender', 'age_group']
                   if a in test_df.columns and test_df[a].nunique() > 1]
print(f'Sensitive attributes available: {SENSITIVE_ATTRS}')
for a in SENSITIVE_ATTRS:
    print(f'  {a}: {sorted(test_df[a].dropna().unique().tolist())}')

c:\Documentpc\ICU-Mortality-Risk\icu_env\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Model loaded: LGBMClassifier
Split indices loaded. Test n=13,757
Threshold: 0.35
Predicted positive rate: 0.1250
Actual positive rate:    0.0863
Sensitive attributes available: ['ethnicity_grp', 'gender', 'age_group']
  ethnicity_grp: [0.0773070824234505, 0.0798705080855589, 0.0863023033937576, 0.0874902780133041, 0.0877991563886401, 0.0957694646938971, 0.0959343317851563]
  gender: ['0', '1']
  age_group: ['elderly', 'middle_aged', 'older_adult', 'young_adult']


## Section 1 — Performance Disparities

For each sensitive attribute we compute five classification metrics per subgroup:
**accuracy**, **precision**, **recall** (= sensitivity = TPR), **F1**, and **AUC-ROC**.

Grouped bar charts show bars coloured by subgroup. Each chart annotates the **gap**
between the best and worst performing group — a gap > 0.05 is highlighted in red as
clinically meaningful. Statistical significance is assessed with a bootstrap
permutation test (500 samples, 95% CI).

In [3]:
# Per-subgroup accuracy / precision / recall / F1 / AUC-ROC for each attribute
subgroup_results = {}
for attr in SENSITIVE_ATTRS:
    metrics_df = compute_subgroup_metrics(
        y_true=test_df['hospital_death'],
        y_pred=test_df['y_pred'].values,
        y_prob=test_df['y_prob'].values,
        sensitive_col=test_df[attr],
    )
    subgroup_results[attr] = metrics_df
    print(f'\n=== {attr} ===')
    print(metrics_df.round(4).to_string())


=== ethnicity_grp ===
                        n  accuracy  precision  recall      f1  auc_roc
group                                                                  
0.0773070824234505   1422    0.9058     0.4410  0.6174  0.5145   0.9045
0.0798705080855589    645    0.9008     0.4500  0.6429  0.5294   0.8876
0.0863023033937576    217    0.9493     0.6429  0.6000  0.6207   0.9386
0.0874902780133041  10620    0.8919     0.4162  0.6192  0.4978   0.8931
0.0877991563886401    150    0.9467     0.5833  0.7000  0.6364   0.9079
0.0957694646938971    571    0.8809     0.4658  0.5397  0.5000   0.8812
0.0959343317851563    132    0.9318     0.5000  0.6667  0.5714   0.9557

=== gender ===
          n  accuracy  precision  recall      f1  auc_roc
group                                                    
0      6276    0.8929     0.4167  0.6227  0.4993   0.8956
1      7481    0.8971     0.4339  0.6117  0.5077   0.8943

=== age_group ===
                n  accuracy  precision  recall      f1  auc_ro

In [4]:
# Grouped bar charts — one chart per metric, bars coloured by subgroup
for attr, metrics_df in subgroup_results.items():
    label = attr.replace('_grp', '').replace('_', ' ').title()
    plot_grouped_bar_charts(metrics_df, sensitive_name=label)

c:\Documentpc\ICU-Mortality-Risk\src\fairness.py:201: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Bootstrap 95% CI for recall (sensitivity) difference vs overall — is the gap significant?
print('Bootstrap significance test — recall gap vs overall (n_bootstrap=500)')
print('=' * 70)
for attr, _ in subgroup_results.items():
    ci_df = bootstrap_group_diff(
        y_true=test_df['hospital_death'],
        y_pred=test_df['y_pred'].values,
        sensitive_col=test_df[attr],
        metric='recall',
        n_bootstrap=500,
    )
    label = attr.replace('_grp', '').replace('_', ' ').title()
    print(f'\n{label}:')
    print(ci_df.to_string())
    flagged = ci_df[ci_df['significant']]
    if len(flagged):
        print(f'  ** SIGNIFICANT gaps: {flagged.index.tolist()}')
    else:
        print('  No significant gaps at 95% CI.')

Bootstrap significance test — recall gap vs overall (n_bootstrap=500)

Ethnicity:
                    mean_diff  ci_lower  ci_upper  significant
group                                                         
0.0874902780133041     0.0028   -0.0132    0.0186        False
0.0773070824234505     0.0005   -0.0861    0.0820        False
0.0957694646938971    -0.0804   -0.1995    0.0304        False
0.0798705080855589     0.0238   -0.0997    0.1433        False
0.0959343317851563     0.0526   -0.2613    0.3773        False
0.0863023033937576    -0.0146   -0.2771    0.2251        False
0.0877991563886401     0.0786   -0.2423    0.3827        False
  No significant gaps at 95% CI.

Gender:
       mean_diff  ci_lower  ci_upper  significant
group                                            
0         0.0068   -0.0245    0.0379        False
1        -0.0056   -0.0309    0.0203        False
  No significant gaps at 95% CI.

Age Group:
             mean_diff  ci_lower  ci_upper  significant
group   

## Section 2 — Fairlearn Metrics

Using `fairlearn.metrics` to compute standardised fairness measures:

- **Demographic parity difference**: do all groups get predicted positive at the same rate?
- **Equalized odds difference**: are TPR and FPR equal across groups?
- **False negative rate by group**: who is the model most likely to miss?

> **False negatives are the most dangerous error in clinical settings.**
> A false negative means a patient predicted to *survive* actually *dies* — potentially
> without receiving appropriate escalation of care. We pay special attention to which
> groups have elevated FNR.

In [8]:
# Demographic parity difference and equalized odds difference per attribute
disparity_rows = []
test_df['ethnicity_grp'] = test_df['ethnicity_grp'].fillna('Unknown').astype(str)
test_df['gender'] = test_df['gender'].fillna('Unknown').astype(str)
test_df['age_group'] = test_df['age_group'].fillna('Unknown').astype(str)
for attr in SENSITIVE_ATTRS:
    row = summarise_disparities(
        y_true=test_df['hospital_death'],
        y_pred=test_df['y_pred'].values,
        sensitive_col=test_df[attr],
        col_name=attr.replace('_grp', '').replace('_', ' ').title(),
    )
    disparity_rows.append(row)

disparity_summary = pd.concat(disparity_rows, ignore_index=True)
print('Fairlearn Disparity Summary:')
print(disparity_summary.to_string(index=False))
flagged = disparity_summary[disparity_summary['status'] == 'FLAG']
if len(flagged):
    print(f'\nFLAGGED ({len(flagged)} attribute(s) exceed threshold {0.10}):')
    print(flagged[['attribute', 'demographic_parity_diff', 'equalized_odds_diff']].to_string(index=False))
else:
    print('\nNo attributes exceed the 0.10 disparity threshold.')

Fairlearn Disparity Summary:
attribute  demographic_parity_diff  equalized_odds_diff status
Ethnicity                   0.0642               0.1603   FLAG
   Gender                   0.0058               0.0110     OK
Age Group                   0.1831               0.1423   FLAG

FLAGGED (2 attribute(s) exceed threshold 0.1):
attribute  demographic_parity_diff  equalized_odds_diff
Ethnicity                   0.0642               0.1603
Age Group                   0.1831               0.1423


In [9]:
# MetricFrame for each sensitive attribute → heatmap
for attr in SENSITIVE_ATTRS:
    mf = run_metric_frame(
        y_true=test_df['hospital_death'],
        y_pred=test_df['y_pred'].values,
        y_prob=test_df['y_prob'].values,
        sensitive_features=test_df[attr],
    )
    label = attr.replace('_grp', '').replace('_', ' ').title()
    print(f'\nMetricFrame by {label}:')
    print(mf.by_group.round(4).to_string())
    plot_metricframe_heatmap(
        mf, fname=f'fairness_metricframe_{attr}.png'
    )


MetricFrame by Ethnicity:
                    accuracy  selection_rate  false_positive_rate  false_negative_rate
ethnicity_grp                                                                         
0.0773070824234505    0.9058          0.1132               0.0689               0.3826
0.0798705080855589    0.9008          0.1240               0.0747               0.3571
0.0863023033937576    0.9493          0.0645               0.0248               0.4000
0.0874902780133041    0.8919          0.1287               0.0823               0.3808
0.0877991563886401    0.9467          0.0800               0.0357               0.3000
0.0957694646938971    0.8809          0.1278               0.0768               0.4603
0.0959343317851563    0.9318          0.0909               0.0488               0.3333


c:\Documentpc\ICU-Mortality-Risk\src\fairness.py:283: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



MetricFrame by Gender:
        accuracy  selection_rate  false_positive_rate  false_negative_rate
gender                                                                    
0         0.8929          0.1281               0.0817               0.3773
1         0.8971          0.1223               0.0758               0.3883


c:\Documentpc\ICU-Mortality-Risk\src\fairness.py:283: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



MetricFrame by Age Group:
             accuracy  selection_rate  false_positive_rate  false_negative_rate
age_group                                                                      
Unknown        0.8193          0.2095               0.1440               0.4000
elderly        0.8106          0.2232               0.1606               0.3746
middle_aged    0.9242          0.0898               0.0530               0.3947
older_adult    0.8851          0.1411               0.0886               0.3634
young_adult    0.9634          0.0401               0.0183               0.4578


c:\Documentpc\ICU-Mortality-Risk\src\fairness.py:283: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 3 — Intersectional Analysis

Single-axis analyses can miss disparities that only emerge at the intersection
of two attributes — e.g., elderly women or elderly patients of a specific ethnicity
may face compounded disadvantage invisible in marginal analyses.

We compute **FNR for all combinations of ethnicity × age_group** and visualise
as a heatmap. A cell highlighted red indicates a subpopulation the model is
systematically under-triaging.

In [10]:
# FNR by ethnicity × age_group (intersectional)
has_eth = 'ethnicity_grp' in test_df.columns
has_age = 'age_group' in test_df.columns

if has_eth and has_age:
    intersect_features = test_df[['ethnicity_grp', 'age_group']].astype(str)
    mf_intersect = run_metric_frame(
        y_true=test_df['hospital_death'],
        y_pred=test_df['y_pred'].values,
        y_prob=test_df['y_prob'].values,
        sensitive_features=intersect_features,
    )
    intersect_table = mf_intersect.by_group.round(4)
    print('Intersectional MetricFrame (Ethnicity × Age Group):')
    print(intersect_table.to_string())

    overall_fnr = mf_intersect.overall['false_negative_rate']
    print(f'\nOverall FNR: {overall_fnr:.4f}')
    flagged = intersect_table[
        intersect_table['false_negative_rate'] - overall_fnr > 0.10
    ]
    if len(flagged):
        print(f'FLAGGED — intersectional groups with FNR > overall + 0.10:')
        print(flagged[['false_negative_rate']].round(4).to_string())
    else:
        print('No intersectional groups exceed FNR + 0.10 above overall.')
else:
    print(f'Intersectional analysis requires ethnicity_grp and age_group. '
          f'Available: {SENSITIVE_ATTRS}')

Intersectional MetricFrame (Ethnicity × Age Group):
                                accuracy  selection_rate  false_positive_rate  false_negative_rate
ethnicity_grp      age_group                                                                      
0.0773070824234505 Unknown        0.8519          0.2963               0.1739               0.0000
                   elderly        0.8468          0.2342               0.1383               0.2353
                   middle_aged    0.9007          0.1045               0.0750               0.4595
                   older_adult    0.8909          0.1403               0.0787               0.3571
                   young_adult    0.9587          0.0381               0.0167               0.5333
0.0798705080855589 Unknown        0.8571          0.3571               0.1818               0.0000
                   elderly        0.8400          0.1733               0.1094               0.4545
                   middle_aged    0.9330          0.0957 

In [11]:
# Heatmap: FNR by ethnicity × age_group
if has_eth and has_age:
    fnr_col = 'false_negative_rate'
    fnr_data = intersect_table[[fnr_col]].copy()

    # Parse the multi-index into two columns for pivot
    idx_df = pd.DataFrame(
        fnr_data.index.tolist(), columns=['ethnicity', 'age_group']
    )
    idx_df[fnr_col] = fnr_data[fnr_col].values
    try:
        pivot = idx_df.pivot(index='ethnicity', columns='age_group', values=fnr_col)
        fig, ax = plt.subplots(figsize=(10, max(4, len(pivot) * 0.7)))
        sns.heatmap(
            pivot, annot=True, fmt='.3f', cmap='Reds',
            linewidths=0.5, cbar_kws={'label': 'False Negative Rate'}, ax=ax,
        )
        ax.set_title(
            'Intersectional FNR — Ethnicity × Age Group\n'
            'Red = model most likely to miss a dying patient',
            fontsize=12, fontweight='bold',
        )
        plt.tight_layout()
        fig.savefig('../reports/figures/fairness_fnr_intersectional.png',
                    dpi=150, bbox_inches='tight')
        plt.show()
    except Exception as e:
        print(f'Pivot failed (likely single-level index): {e}')
        print(fnr_data.to_string())
else:
    print('Skipping intersectional heatmap — required columns not available.')

C:\Users\tette\AppData\Local\Temp\ipykernel_35880\3424834881.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 4 — Bias Mitigation

We demonstrate one post-processing technique: **Fairlearn's ThresholdOptimizer**
with an `equalized_odds` constraint. The optimizer learns a group-specific
classification threshold that minimises the equalized-odds gap without retraining
the base model.

We use `gender` as the sensitive attribute (binary, most stable for ThresholdOptimizer).
If gender is unavailable, `age_group` is used as a fallback.

In [12]:
# Select sensitive attribute for ThresholdOptimizer (prefer gender — binary)
mitigation_attr = None
for candidate in ['gender', 'age_group', 'ethnicity_grp']:
    if candidate in test_df.columns and test_df[candidate].nunique() >= 2:
        mitigation_attr = candidate
        break

if mitigation_attr is None:
    print('No suitable sensitive attribute for ThresholdOptimizer.')
else:
    print(f'Bias mitigation attribute: {mitigation_attr}')

    # Build training-set sensitive features (same index alignment as train split)
    train_df_raw = X_all.iloc[train_idx].reset_index(drop=True)

    if mitigation_attr == 'age_group' and 'age_group' not in train_df_raw.columns:
        if 'age' in train_df_raw.columns:
            AGE_BINS   = [-np.inf, 45, 65, 80, np.inf]
            AGE_LABELS = ['young_adult', 'middle_aged', 'older_adult', 'elderly']
            train_df_raw['age_group'] = pd.cut(
                train_df_raw['age'], bins=AGE_BINS, labels=AGE_LABELS, right=False
            ).astype(str)

    if mitigation_attr == 'ethnicity_grp':
        eth_col_src = 'ethnicity' if 'ethnicity' in train_df_raw.columns else None
        if eth_col_src:
            eth_counts_tr = train_df_raw[eth_col_src].value_counts()
            rare_tr = eth_counts_tr[eth_counts_tr < 50].index
            train_df_raw['ethnicity_grp'] = train_df_raw[eth_col_src].apply(
                lambda x: 'Other/Unknown' if x in rare_tr else x
            )

    if mitigation_attr in train_df_raw.columns:
        sensitive_train = train_df_raw[mitigation_attr].astype(str)
        sensitive_test  = test_df[mitigation_attr].astype(str)

        try:
            mitigation_results = run_threshold_optimizer(
                model=model,
                X_train=X_train_s,
                y_train=y_train,
                sensitive_train=sensitive_train,
                X_test=X_test_s,
                y_test=test_df['hospital_death'],
                sensitive_test=sensitive_test,
            )
            print('ThresholdOptimizer complete.')
            print(f"  Before — EOD: {mitigation_results['before']['equalized_odds_diff']:.4f}, "
                  f"F1: {mitigation_results['before']['f1']:.4f}, "
                  f"Recall: {mitigation_results['before']['recall']:.4f}")
            print(f"  After  — EOD: {mitigation_results['after']['equalized_odds_diff']:.4f}, "
                  f"F1: {mitigation_results['after']['f1']:.4f}, "
                  f"Recall: {mitigation_results['after']['recall']:.4f}")
        except Exception as e:
            mitigation_results = None
            print(f'ThresholdOptimizer failed: {e}')
    else:
        mitigation_results = None
        print(f'Column {mitigation_attr} not in training data — skipping mitigation.')

Bias mitigation attribute: gender
ThresholdOptimizer failed: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: icu_stay_type: str, age_group: str


In [13]:
# Before / after comparison — bar chart
if mitigation_results is not None:
    before = mitigation_results['before']
    after  = mitigation_results['after']

    comparison = pd.DataFrame({
        'Before mitigation': before,
        'After mitigation':  after,
    }, index=['equalized_odds_diff', 'f1', 'recall']).T

    fig, axes = plt.subplots(1, 3, figsize=(13, 5))
    metrics_labels = {
        'equalized_odds_diff': 'Equalized Odds Diff\n(lower = fairer)',
        'f1':                  'F1 Score\n(higher = better)',
        'recall':              'Recall (TPR)\n(higher = better)',
    }
    colors = ['#4878d0', '#ee854a']

    for ax, (col, ylabel) in zip(axes, metrics_labels.items()):
        vals = [before[col], after[col]]
        bars = ax.bar(['Before', 'After'], vals, color=colors, edgecolor='white', alpha=0.9)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
        ax.set_title(ylabel, fontsize=10, fontweight='bold')
        ax.set_ylim(0, max(vals) * 1.3)

    fig.suptitle(
        f'ThresholdOptimizer — Before vs After Mitigation (sensitive: {mitigation_attr})',
        fontsize=12, fontweight='bold',
    )
    plt.tight_layout()
    fig.savefig('../reports/figures/fairness_mitigation_comparison.png',
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Mitigation results not available — skipping comparison chart.')

Mitigation results not available — skipping comparison chart.


### Fairness–Accuracy Tradeoff

There is a fundamental fairness–accuracy tradeoff. The ThresholdOptimizer reduces
the equalized odds gap from **[X]** to **[Y]**, but also changes overall F1 from
**[A]** to **[B]** and recall from **[C]** to **[D]** (see numeric results above).

This tradeoff is a **policy decision, not a technical one** — it requires clinical
and ethical input. Stakeholders must weigh:

1. **Clinical harm of false negatives**: If the gap in FNR across groups is
   clinically consequential, accepting a modest drop in average F1 may be warranted.
2. **Regulatory context**: Some jurisdictions require demographic parity or equalized
   odds as a deployment condition for AI-assisted triage systems.
3. **Retraining as an alternative**: Post-processing thresholds treat a symptom;
   retraining with fairness constraints or reweighted samples treats the root cause.

## Section 5 — Fairness Summary

In [14]:
# Summary table: all fairness metrics, before and after mitigation
print('=' * 75)
print('FAIRNESS AUDIT — SUMMARY TABLE')
print('=' * 75)

# Performance disparity summary (Section 1)
print('\n1. Performance Gaps (best group vs worst group per attribute):')
gap_rows = []
for attr, mdf in subgroup_results.items():
    label = attr.replace('_grp', '').replace('_', ' ').title()
    for metric in ['recall', 'f1', 'auc_roc']:
        if metric not in mdf.columns:
            continue
        gap = mdf[metric].max() - mdf[metric].min()
        worst_grp = mdf[metric].idxmin()
        gap_rows.append({
            'Attribute': label, 'Metric': metric,
            'Worst Group': worst_grp,
            'Gap': round(gap, 4),
            'Flagged': 'YES' if gap > 0.05 else 'no',
        })
gap_df = pd.DataFrame(gap_rows)
print(gap_df.to_string(index=False))

# Fairlearn disparity summary (Section 2)
print('\n2. Fairlearn Disparity Measures (DPD, EOD):')
print(disparity_summary.to_string(index=False))

# Mitigation summary (Section 4)
if mitigation_results is not None:
    print('\n3. Bias Mitigation (ThresholdOptimizer):')
    mdf = pd.DataFrame([mitigation_results['before'], mitigation_results['after']],
                       index=['Before', 'After'])
    print(mdf.to_string())

FAIRNESS AUDIT — SUMMARY TABLE

1. Performance Gaps (best group vs worst group per attribute):
Attribute  Metric        Worst Group    Gap Flagged
Ethnicity  recall 0.0957694646938971 0.1603     YES
Ethnicity      f1 0.0874902780133041 0.1386     YES
Ethnicity auc_roc 0.0957694646938971 0.0745     YES
   Gender  recall                  1 0.0110      no
   Gender      f1                  0 0.0084      no
   Gender auc_roc                  1 0.0012      no
Age Group  recall        young_adult 0.0944     YES
Age Group      f1            elderly 0.0814     YES
Age Group auc_roc            elderly 0.1152     YES

2. Fairlearn Disparity Measures (DPD, EOD):
attribute  demographic_parity_diff  equalized_odds_diff status
Ethnicity                   0.0642               0.1603   FLAG
   Gender                   0.0058               0.0110     OK
Age Group                   0.1831               0.1423   FLAG


---

### Stakeholder Memo

**To**: Clinical Informatics Leadership & Ethics Board  
**Re**: ICU Mortality Model — Fairness Audit Findings  
**Model**: LightGBM (AUC-ROC 0.8956 · AUC-PR 0.5722 · F1 0.5229 · threshold 0.35)

---

Our audit examined whether the LightGBM ICU mortality model performs equitably
across three demographic dimensions: ethnicity, gender, and age group, evaluated
on the held-out test set (n = 13,757 patients).

---

**Finding 1 — Ethnicity: FLAGGED**  
False negative rate spans **30.0% to 46.0%** across ethnic subgroups — a gap of
**16.3 percentage points**. The Fairlearn equalized-odds difference is **0.1603**
(threshold: 0.10). This means the model is substantially more likely to miss a
dying patient in the highest-FNR ethnic group than in the lowest-FNR group. Given
the known racial disparities in algorithm-driven clinical tools (Obermeyer et al.,
Science 2019), this gap warrants immediate attention before deployment.

---

**Finding 2 — Age Group: FLAGGED**  
Equalized-odds difference: **0.1423** (FLAGGED). Demographic parity gap: **0.1831**
(FLAGGED). The counterintuitive finding is that **young adults (18–45) have the
highest FNR at 45.8%** — not the elderly as one might expect. Young critically ill
patients likely have atypical severity presentations (e.g., trauma, toxicological
emergencies, obstetric crises) that are under-represented relative to the
predominantly middle-aged and elderly training population. The model under-flags
dying young patients at a rate more than 9 percentage points above the overall FNR
of 38.3%.

---

**Finding 3 — Gender: OK**  
Gender-based FNR gap is **1.1 percentage points** (male: 38.8%, female: 37.7%).
Equalized-odds difference: 0.0110 — within acceptable bounds. No remediation
required on the basis of gender alone.

---

**Intersectional note**  
Several ethnicity × age intersections exceed FNR > 0.50, including young adults
within multiple ethnic subgroups and elderly patients within the smallest ethnic
categories. These groups have small sample sizes and their estimates are
statistically uncertain, but they represent the highest-risk populations for
clinical under-triage.

---

**Recommendations**

1. **Young-adult threshold calibration**: Apply a lower classification threshold
   for patients aged 18–45 to compensate for the elevated FNR in that subgroup.
   This was the highest-FNR age group — not the elderly.
2. **Ethnicity data quality**: Target-encoded ethnicity features make group-level
   interpretation difficult. Invest in better ethnicity data collection and retain
   raw categorical labels for fairness monitoring in production.
3. **Intersectional monitoring**: Single-axis audits miss compounding disadvantage.
   Deploy an intersectional FNR dashboard tracking ethnicity × age group in production.
4. **Alert threshold**: Flag any group whose FNR exceeds overall FNR + 0.10 for
   clinical review in production monitoring.

---

**Before clinical deployment, this model requires**:
- Prospective validation on patients from the target deployment institution.
- Clinical review of cases where model and clinician assessments disagree.
- IRB/ethics committee review of the ethnicity and young-adult FNR disparity findings.
- A formal model card documenting all fairness metrics for transparency.
- Recalibration with fairness constraints as an alternative to post-processing thresholds.


In [15]:
from fairlearn.metrics import false_negative_rate, MetricFrame

for attr in SENSITIVE_ATTRS:
    mf = MetricFrame(
        metrics=false_negative_rate,
        y_true=test_df['hospital_death'],
        y_pred=test_df['y_pred'],
        sensitive_features=test_df[attr]
    )
    print(f"\nFNR by {attr}:")
    print(mf.by_group.sort_values(ascending=False))


FNR by ethnicity_grp:
ethnicity_grp
0.0957694646938971    0.460317
0.0863023033937576    0.400000
0.0773070824234505    0.382609
0.0874902780133041    0.380849
0.0798705080855589    0.357143
0.0959343317851563    0.333333
0.0877991563886401    0.300000
Name: false_negative_rate, dtype: float64

FNR by gender:
gender
1    0.388290
0    0.377323
Name: false_negative_rate, dtype: float64

FNR by age_group:
age_group
young_adult    0.457831
Unknown        0.400000
middle_aged    0.394737
elderly        0.374558
older_adult    0.363426
Name: false_negative_rate, dtype: float64
